BÀI THỰC HÀNH MÔ HÌNH NGÔN NGỮ

Nội dung:

1) Mô hình ngôn ngữ N-GRAM:

- Huấn luyện mô hình ngôn ngữ với ngữ liệu ham và spam.

- Dùng mô hình để tạo sinh các câu ham và spam.

- Dùng mô hình phân lớp được huấn luyện trên ngữ liệu ham và spam để đánh giá trên tập câu đã được tạo sinh.

2) Đánh giá mô hình ngôn ngữ:
Dùng mô hình để tính perplexity trên ngữ liệu test.

In [ ]:
%%capture
!pip install --upgrade kagglehub
!pip install --upgrade scikit-learn

CHUẨN BỊ NGỮ LIỆU

Chúng ta sẽ dùng ngữ liệu sms-spam

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import random
import kagglehub
import nltk
from nltk import bigrams, trigrams
from collections import defaultdict
import os


In [ ]:
nltk.download('punkt_tab')
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("!cp {}/spam.csv /content/".format(path))
os.system("cp {}/spam.csv /content/".format(path))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
!cp /kaggle/input/sms-spam-collection-dataset/spam.csv /content/


0

In [ ]:
dataset = pd.read_csv("spam.csv", encoding="windows-1252")
dataset.info()
dataset = dataset.drop(dataset[["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"]], axis=1)
dataset.rename(columns = {"v1":"Class", "v2":"Message"}, inplace = True)
dataset.head()

msgs = list(dataset["Message"])
clss = list(dataset["Class"])
hamtext = []
spamtext = []
VOCAB = {"<START>": 0, "<END>": 1}
n = 0
for i in range(len(msgs)):
  for w in nltk.word_tokenize(msgs[i]):
    if VOCAB.get(w) == None:
      VOCAB[w] = n
      n += 1
  if clss[i] == "ham":
    hamtext.append(msgs[i])
  else:
    spamtext.append(msgs[i])

DICT = list(VOCAB.keys())

n = round(len(hamtext) * 0.8)
hamTrain = hamtext[:n]
hamTest = hamtext[n:]

n = round(len(spamtext) * 0.8)
spamTrain = spamtext[:n]
spamTest = spamtext[n:]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


I) Mô hình ngôn ngữ N-gram

Trong ví dụ này, chúng ta dùng mô hình ngôn ngữ Bi-gram.

Chúng ta cài đặt hàm tạo mô hình ngôn ngữ Bi-gram và hàm tạo sinh câu từ mô hình ngôn ngữ Bi-gram.

In [ ]:
def buildBigramModel(t, vocab, alpha):
  size = len(vocab)
  model = np.zeros([size, size])
  for sent in t:
    grams = ['<START>'] + nltk.word_tokenize(sent) + ['<END>']
    bg = list(bigrams(grams))
    for w1, w2 in bg:
      model[vocab[w1]][vocab[w2]] += 1

  for i in range(size):
    N = sum(model[i]) + alpha * size
    for j in range(size):
      model[i][j] = (model[i][j] + alpha) / N

  return model

def generate(model, vocab, dictionary, word, step, nstep):
  if step == nstep:
    return "<END>"
  d = model[vocab[word]]
  size = len(model)
  c = 0
  ws = []
  ps = []
  for i in range(size):
    c += d[i]
    ps.append(c)
  if c == 0:
    return "<END>"

  p = random.uniform(0,1)
  for i in range(len(ps)):
    if ps[i] >= p:
      break
  nextWord = ""
  if i == range(len(ps)):
    nextWord = dictionary[-1]
  else:
    nextWord = dictionary[i]
  if nextWord == "<END>":
    return nextWord
  return "{} {}".format(nextWord, generate(model, VOCAB, dictionary, nextWord, step+1, nstep ))



Huấn luyện hai mô hình ngôn ngữ cho ham và spam

In [ ]:
hamModel = buildBigramModel(hamTrain, VOCAB, 0.001)
spamModel = buildBigramModel(spamTrain, VOCAB, 0.001)

In [ ]:
print(hamModel[-2])
print(sum(hamModel[-2]))

[8.67904878e-05 8.67904878e-05 8.67904878e-05 ... 8.67904878e-05
 8.67904878e-05 8.67904878e-05]
0.9999999999997883


Tạo sinh 10 câu cho ngữ liệu ham và 10 câu cho ngữ liệu spam từ các mô hình đã tạo được.

In [ ]:

print("----------------- Ham messages -----------------")
for i in range(10):
  result = generate(hamModel, VOCAB, DICT, "<START>", 1, 10)
  print(result)
print("----------------- Spam messages -----------------")
for i in range(10):
  result = generate(spamModel, VOCAB, DICT, "<START>", 1, 10)
  print(result)

----------------- Ham messages -----------------
situation wins Belovd quizzes appt walsall \Getting bear WOTU <END>
puttin class Available plate bffs generally tactless opinion Childrens <END>
situation smells 2NITE EVENING upgrading ,s ogunrinde runs Abbey <END>
these B-Blue BEST pounded phoned creep Co Warner Smith-Switch <END>
evil welp guilty courage Forever re 12:30 maths us <END>
Work plum garments adventuring 69866.18 Reverse Otherwise response wkly <END>
Yes bye dudes Yalru drink.pa Ge fgkslpoPW plan wkly <END>
census ni8 cons Wife God Vettam ago network brother <END>
basically bad- snickering cuz bian Bennys supply MAKE qatar <END>
then home Caller xxxxxxx\ mates as for www.rtf.sphosting.com Symbol <END>
----------------- Spam messages -----------------
dearly March greece spys upcharge www.bridal.petticoatdreams.co.uk nannys 1Apple/Day=No AMY <END>
MSG Jackpot Wasted GM batchlor Anything virgins STaY å£4.50 <END>
secretly PO19 drms.take juan Freeentry 0721072 handed waheed t

Tính toán perplexity của từng mô hình trên các ngữ liệu test.

In [ ]:
def perplexity(mdl, test):
  z = 0
  n = 0
  for s in test:
    pre = "<START>"
    for word in nltk.word_tokenize(s):
      z = z + math.log(mdl[VOCAB[pre]][VOCAB[word]])
      n += 1
      pre = word
    z = z + math.log(mdl[VOCAB[pre]][VOCAB["<END>"]])
  return math.exp(-z/n)


In [ ]:
print("Ham Model on Spam train:",perplexity(hamModel, spamTrain))
print("Spam Model on Spam train:",perplexity(spamModel, spamTrain))


Ham Model on Ham train: 17422.49741082518
Spam Model on Spam train: 19.934073272163968


Bài tập: Các bạn hãy làm lại bài tập này với việc kết hợp các yêu cầu sau:
- Dùng các phương pháp smoothing khác Additive alpha.
- Dùng mô hình ngôn ngữ uni-gram, tri-gram, hoặc kết hợp uni-gram + bi-gram, bi-gram + tri-gram, ...
- Lưu ý trường hợp Out of Vocabulary